# 01: Data Preprocessing

### Data Download for Clinical, Microenvironment, and Expression data
1. Go to https://portal.gdc.cancer.gov/analysis_page?app=CohortBuilder&tab=general
    - Select Program: TCGA
    - "# CASES" -> "Table View" -> "TSV"
    - Download to `data_raw/`
2. Go to "Repository" with the previous TCGA selection still active.
2. Make the following selections: 
    - Experimental Strategy: RNA-Seq
    - Access: open
    - Tumor Descriptor: primary, not applicable
    - Specimen Type: solid tissue
    - Preservation Method: unknown
3. "Add All Files to Cart". Go to "Cart". 
4. Place the following downloads in the `data_raw/unknown` folder.
    - Download Associated Data
        - Clinical: TSV
        - Biospecimen: TSV
        - Sample Sheet
    - Unzip these files
5. Place the following downloads in the `data_raw/unknown/rnaseq` folder.
    - Download Cart
        - Manifest
4. "Remove from Cart" -> "All Files"
4. Go back to downloads, select the same filters but change
    - Preservation Method: all except unknown (oct and ffpe)
5. Repeat the download, placing in `data_raw/other` and `data_raw/other/rnaseq`.
6. Use the GDC client CLI to download both manifests (~60GB)
```
cd data_raw/unknown/rnaseq
gdc-client download -m ../<your_unknown_manifest>.txt
cd ../../other/rnaseq
gdc-client download -m ../<your_other_manifest>.txt
```

### Data Download for SNV and SCNA data
This data is under access controls. Please apply online and refer to the cited studies in our supplementary material for compiling this data.

In [9]:
import os
import pandas as pd
import numpy as np

# Replace these paths to run for both the "unknown" and "other" downloads
in_dir = 'data_raw/other'
out_dir = 'data_intermediate/other'
os.makedirs(out_dir, exist_ok=True)

# Replace these with your folder names
tcga_cases_tsv = 'data_raw/cases.tsv'
tcga_clinical_tsv = os.path.join(in_dir, 'clinical.cart.2025-01-17/clinical.tsv')
tcga_sample_tsv = os.path.join(in_dir, 'biospecimen.cart.2025-01-17/sample.tsv')
tcga_slide_tsv = os.path.join(in_dir, 'biospecimen.cart.2025-01-17/slide.tsv')
tcga_file_tsv = os.path.join(in_dir, 'gdc_sample_sheet.2025-01-17.tsv')

In [10]:
tcga_cases_df = pd.read_csv(tcga_cases_tsv, sep='\t')
tcga_clinical_df = pd.read_csv(tcga_clinical_tsv, sep='\t')
tcga_sample_df = pd.read_csv(tcga_sample_tsv, sep='\t')
tcga_slide_df = pd.read_csv(tcga_slide_tsv, sep='\t')
tcga_file_df = pd.read_csv(tcga_file_tsv, sep='\t')

/var/folders/_6/468mk1cx3cb080z2r5gq0k_r0000gn/T/ipykernel_95021/206510135.py:1: DtypeWarning: Columns (11,13,15,17,19,21,23,25,27,29,33,35,37,39,41,43,45,47,61) have mixed types. Specify dtype option on import or set low_memory=False.
  tcga_cases_df = pd.read_csv(tcga_cases_tsv, sep='\t')


In [11]:
selected_case_cols = [
    'case_id',
    'submitter_id',
    'primary_site',
]
selected_clinical_cols = [
    'case_id',
    'case_submitter_id',
    'project_id',
    'age_at_diagnosis',
    'gender',
    'year_of_birth',
    'race',
    'ajcc_pathologic_stage',
    'ajcc_clinical_stage',
    'ann_arbor_clinical_stage',
    'figo_stage',
]
selected_survival_cols = [
    'case_id',
    'case_submitter_id',
    'vital_status',
    'days_to_death',
    'days_to_last_follow_up',
]
selected_sample_cols = [
    'case_id',
    'case_submitter_id',
    'sample_id',
    'sample_submitter_id',
    'days_to_collection',
    'sample_type',
]
selected_slide_cols = [
    'case_id',
    'case_submitter_id',
    'sample_id',
    'sample_submitter_id',
    'slide_submitter_id',
    'percent_neutrophil_infiltration',
    'percent_monocyte_infiltration',
    'percent_normal_cells',
    'percent_tumor_nuclei',
    'percent_lymphocyte_infiltration',
    'percent_stromal_cells',
    'percent_tumor_cells',
]
study_names = {
    "LAML": "Acute Myeloid Leukemia",
    "ACC": "Adrenocortical carcinoma",
    "BLCA": "Bladder Urothelial Carcinoma",
    "LGG": "Brain Lower Grade Glioma",
    "BRCA": "Breast invasive carcinoma",
    "CESC": "Cervical squamous cell carcinoma and endocervical adenocarcinoma",
    "CHOL": "Cholangiocarcinoma",
    "LCML": "Chronic Myelogenous Leukemia",
    "COAD": "Colon adenocarcinoma",
    "CNTL": "Controls",
    "ESCA": "Esophageal carcinoma",
    "FPPP": "FFPE Pilot Phase II",
    "GBM": "Glioblastoma multiforme",
    "HNSC": "Head and Neck squamous cell carcinoma",
    "KICH": "Kidney Chromophobe",
    "KIRC": "Kidney renal clear cell carcinoma",
    "KIRP": "Kidney renal papillary cell carcinoma",
    "LIHC": "Liver hepatocellular carcinoma",
    "LUAD": "Lung adenocarcinoma",
    "LUSC": "Lung squamous cell carcinoma",
    "DLBC": "Lymphoid Neoplasm Diffuse Large B-cell Lymphoma",
    "MESO": "Mesothelioma",
    "MISC": "Miscellaneous",
    "OV": "Ovarian serous cystadenocarcinoma",
    "PAAD": "Pancreatic adenocarcinoma",
    "PCPG": "Pheochromocytoma and Paraganglioma",
    "PRAD": "Prostate adenocarcinoma",
    "READ": "Rectum adenocarcinoma",
    "SARC": "Sarcoma",
    "SKCM": "Skin Cutaneous Melanoma",
    "STAD": "Stomach adenocarcinoma",
    "TGCT": "Testicular Germ Cell Tumors",
    "THYM": "Thymoma",
    "THCA": "Thyroid carcinoma",
    "UCS": "Uterine Carcinosarcoma",
    "UCEC": "Uterine Corpus Endometrial Carcinoma",
    "UVM": "Uveal Melanoma",
}

In [12]:
print('SURVIVAL')
tcga_survival_df = tcga_clinical_df[selected_survival_cols]
display(tcga_survival_df.head())
print('CASES')
tcga_cases_df = tcga_cases_df[selected_case_cols]
display(tcga_cases_df.head())
print('CLINICAL')
tcga_clinical_df = tcga_clinical_df[selected_clinical_cols]
display(tcga_clinical_df.head())
print('SAMPLE')
tcga_sample_df = tcga_sample_df[selected_sample_cols]
display(tcga_sample_df.head())
print('SLIDE')
tcga_slide_df = tcga_slide_df[selected_slide_cols]
display(tcga_slide_df.head())
print('RNASEQ FILES')
tcga_file_df.drop_duplicates(subset='Sample ID', inplace=True, keep='last')
display(tcga_file_df.head())

SURVIVAL


,case_id,case_submitter_id,vital_status,days_to_death,days_to_last_follow_up
0,0004d251-3f70-4395-b175-c94c2f5b1b81,TCGA-DD-AAVP,Alive,'--,2752.0
1,0004d251-3f70-4395-b175-c94c2f5b1b81,TCGA-DD-AAVP,Alive,'--,2752.0
2,000d566c-96c7-4f1c-b36e-fa2222467983,TCGA-KK-A7B2,Alive,'--,1099.0
3,000d566c-96c7-4f1c-b36e-fa2222467983,TCGA-KK-A7B2,Alive,'--,1099.0
4,001887aa-36d0-463f-8bca-dec7043b4f2e,TCGA-DD-A4NP,Alive,'--,3308.0


CASES


,case_id,submitter_id,primary_site
0,4298ccdb-2e6d-4267-822d-75b021364084,TCGA-B0-4710,Kidney
1,a2663a86-a006-4867-9e88-2b523df48303,TCGA-B8-A54K,Kidney
2,439794a8-51bd-4c70-968c-34cf26b90148,TCGA-B0-5113,Kidney
3,e865d40a-9989-436c-8426-88cc84c863e8,TCGA-CJ-5689,Kidney
4,305eaef4-4644-46e3-a696-d2e4a972f691,TCGA-CZ-4865,Kidney


CLINICAL


...

,case_id,case_submitter_id,sample_id,sample_submitter_id,days_to_collection,sample_type
0,00fd9306-4a68-49ab-a768-e5fed126a765,TCGA-NC-A5HJ,2d19b480-68f0-41ae-b7b0-2dfab1eda121,TCGA-NC-A5HJ-10A,1213,Blood Derived Normal
1,00fd9306-4a68-49ab-a768-e5fed126a765,TCGA-NC-A5HJ,70601874-9d07-4c3a-b0e0-57757e6fa4a0,TCGA-NC-A5HJ-01Z,'--,Primary Tumor
2,00fd9306-4a68-49ab-a768-e5fed126a765,TCGA-NC-A5HJ,97163b07-0328-4661-a06e-185f7849e819,TCGA-NC-A5HJ-01A,1213,Primary Tumor
3,987e8157-fbea-40f3-aa01-0092d6cceb97,TCGA-XF-AAN2,2853b600-0838-4298-8187-e44a679042d7,TCGA-XF-AAN2-01Z,'--,Primary Tumor
4,987e8157-fbea-40f3-aa01-0092d6cceb97,TCGA-XF-AAN2,41f8f766-2304-4ed4-8777-5d8b027ff9f9,TCGA-XF-AAN2-01A,2505,Primary Tumor


SLIDE


,case_id,case_submitter_id,sample_id,sample_submitter_id,slide_submitter_id,percent_neutrophil_infiltration,percent_monocyte_infiltration,percent_normal_cells,percent_tumor_nuclei,percent_lymphocyte_infiltration,percent_stromal_cells,percent_tumor_cells
0,00fd9306-4a68-49ab-a768-e5fed126a765,TCGA-NC-A5HJ,70601874-9d07-4c3a-b0e0-57757e6fa4a0,TCGA-NC-A5HJ-01Z,TCGA-NC-A5HJ-01Z-00-DX1,'--,'--,'--,'--,'--,'--,'--
1,00fd9306-4a68-49ab-a768-e5fed126a765,TCGA-NC-A5HJ,97163b07-0328-4661-a06e-185f7849e819,TCGA-NC-A5HJ-01A,TCGA-NC-A5HJ-01A-01-TS1,0.0,4.0,0.0,95.0,2.0,10.0,80.0
2,987e8157-fbea-40f3-aa01-0092d6cceb97,TCGA-XF-AAN2,2853b600-0838-4298-8187-e44a679042d7,TCGA-XF-AAN2-01Z,TCGA-XF-AAN2-01Z-00-DX1,'--,'--,'--,'--,'--,'--,'--
3,987e8157-fbea-40f3-aa01-0092d6cceb97,TCGA-XF-AAN2,41f8f766-2304-4ed4-8777-5d8b027ff9f9,TCGA-XF-AAN2-01A,TCGA-XF-AAN2-01A-01-TSA,4.0,1.0,0.0,65.0,2.0,15.0,65.0
4,d18cd259-267e-47b5-9353-ad3454e048a0,TCGA-CC-A8HS,41ca0b98-d4b2-4a09-a5e3-20fedb440926,TCGA-CC-A8HS-01A,TCGA-CC-A8HS-01A-01-TSA,0.0,0.0,0.0,95.0,0.0,0.0,100.0


RNASEQ FILES


,File ID,File Name,Data Category,Data Type,Project ID,Case ID,Sample ID,Sample Type
0,996c69b3-c15a-41bb-8105-f2b4a5b40325,e84e23ef-7bf2-4770-8193-a058dcd4c61a.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-ESCA,TCGA-IC-A6RE,TCGA-IC-A6RE-11A,Solid Tissue Normal
1,a36c99ec-753c-4859-9082-b077434a3f23,efee1a58-1e28-43d2-92bf-37f8b9107312.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-ESCA,TCGA-IG-A97I,TCGA-IG-A97I-01A,Primary Tumor
2,c7985505-df33-4c0e-98bb-188003ea701f,6c975753-106b-45bb-bf7d-146740004db1.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-ESCA,TCGA-IC-A6RE,TCGA-IC-A6RE-01A,Primary Tumor
3,590af49d-2e1d-4cad-ab8c-9abe3c7558b0,40ac3474-e908-43db-861b-98e9bfb61739.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-ESCA,TCGA-IG-A3I8,TCGA-IG-A3I8-01A,Primary Tumor
4,dee51224-652b-4656-902c-f1b18bed8aa1,52c2c5f4-4395-4764-bf22-9fa84bf2f5c2.rna_seq.a...,Transcriptome Profiling,Gene Expression Quantification,TCGA-ESCA,TCGA-L5-A8NT,TCGA-L5-A8NT-01A,Primary Tumor


In [13]:
# Compile clinical covariates
my_clinical_df = tcga_clinical_df.merge(tcga_cases_df, on='case_id')
my_clinical_df['disease_type'] = tcga_clinical_df['project_id'].map(lambda s: s.split('-')[1])
my_clinical_df.drop(columns=['case_submitter_id', 'project_id'], inplace=True)
my_clinical_df.replace("'--", None, inplace=True)

stage_features = [
    'ajcc_pathologic_stage',
    'ajcc_clinical_stage',
    'ann_arbor_clinical_stage',
    'figo_stage',
]
stages = []
for i, (ajcc_path, ajcc_clin, ann, figo) in my_clinical_df[stage_features].iterrows():
    def get_stage(s):
        if "IV" in s:
            return 4.
        elif "III" in s:
            return 3.
        elif "II" in s:
            return 2.
        elif "I" in s:
            return 1.
        else:
            return None
    if ajcc_path is not None:
        stage = get_stage(ajcc_path)
    elif ajcc_clin is not None:
        stage = get_stage(ajcc_clin)
    elif ann is not None:
        stage = get_stage(ann)
    elif figo is not None:
        stage = get_stage(figo)
    else:
        stage = None
    stages.append(stage)
my_clinical_df['stage'] = stages 
my_clinical_df.drop(columns=stage_features, inplace=True)
my_clinical_df.to_csv(os.path.join(out_dir, 'clinical_covariates.csv'), index=False)

my_clinical_df.head()

,case_id,age_at_diagnosis,gender,year_of_birth,race,submitter_id,primary_site,disease_type,stage
0,0004d251-3f70-4395-b175-c94c2f5b1b81,17833,male,1959,asian,TCGA-DD-AAVP,Liver and intrahepatic bile ducts,LIHC,1.0
1,0004d251-3f70-4395-b175-c94c2f5b1b81,17833,male,1959,asian,TCGA-DD-AAVP,Liver and intrahepatic bile ducts,LIHC,1.0
2,000d566c-96c7-4f1c-b36e-fa2222467983,24845,male,1944,white,TCGA-KK-A7B2,Prostate gland,PRAD,NaN
3,000d566c-96c7-4f1c-b36e-fa2222467983,24845,male,1944,white,TCGA-KK-A7B2,Prostate gland,PRAD,NaN
4,001887aa-36d0-463f-8bca-dec7043b4f2e,11838,male,1973,white,TCGA-DD-A4NP,Liver and intrahepatic bile ducts,LIHC,1.0


In [14]:
# Compile samples
# Only use the first tumor and healthy sample from each patient
my_sample_df = tcga_sample_df[tcga_sample_df['sample_submitter_id'].map(lambda s: '-11A' in s or '-01A' in s)]
my_sample_df = my_sample_df.merge(tcga_slide_df.drop(columns=['case_id', 'case_submitter_id', 'sample_submitter_id', 'slide_submitter_id']), on='sample_id')
my_sample_df.drop_duplicates(subset='sample_id', inplace=True)  # Multiple slides, just take the first
my_sample_df.replace("'--", None, inplace=True)
my_sample_df.to_csv(os.path.join(out_dir, 'biopsy_covariates.csv'), index=False)
my_sample_df.head()

,case_id,case_submitter_id,sample_id,sample_submitter_id,days_to_collection,sample_type,percent_neutrophil_infiltration,percent_monocyte_infiltration,percent_normal_cells,percent_tumor_nuclei,percent_lymphocyte_infiltration,percent_stromal_cells,percent_tumor_cells
0,00fd9306-4a68-49ab-a768-e5fed126a765,TCGA-NC-A5HJ,97163b07-0328-4661-a06e-185f7849e819,TCGA-NC-A5HJ-01A,1213,Primary Tumor,0.0,4.0,0.0,95.0,2.0,10.0,80.0
1,987e8157-fbea-40f3-aa01-0092d6cceb97,TCGA-XF-AAN2,41f8f766-2304-4ed4-8777-5d8b027ff9f9,TCGA-XF-AAN2-01A,2505,Primary Tumor,4.0,1.0,0.0,65.0,2.0,15.0,65.0
2,d18cd259-267e-47b5-9353-ad3454e048a0,TCGA-CC-A8HS,41ca0b98-d4b2-4a09-a5e3-20fedb440926,TCGA-CC-A8HS-01A,223,Primary Tumor,0.0,0.0,0.0,95.0,0.0,0.0,100.0
3,f6743ff8-542e-4fde-8ff2-4878fcabffda,TCGA-EJ-A7NJ,b13d4e3d-898e-4339-a506-110b8e803b6e,TCGA-EJ-A7NJ-01A,211,Primary Tumor,0.0,0.0,20.0,60.0,20.0,35.0,43.0
4,b5fba77b-1f50-4e71-95b2-566afba4bdd7,TCGA-A2-A04Q,e17e273d-6f53-4419-9f0c-1e4e667c5257,TCGA-A2-A04Q-01A,2260,Primary Tumor,0.0,0.0,0.0,80.0,1.0,25.0,73.0


In [15]:
# Survival
my_survival_df = tcga_survival_df[['case_id', 'case_submitter_id', 'vital_status', 'days_to_death', 'days_to_last_follow_up']]
my_survival_df['died'] = my_survival_df['vital_status'] == 'Dead'
my_survival_df['time'] = my_survival_df['days_to_death']
my_survival_df.replace("'--", None, inplace=True)
my_survival_df['time'].fillna(my_survival_df['days_to_last_follow_up'], inplace=True)
my_survival_df.drop(columns=['vital_status', 'days_to_death', 'days_to_last_follow_up'], inplace=True)
my_survival_df.drop_duplicates(inplace=True)
my_survival_df.to_csv(os.path.join(out_dir, 'survival.csv'), index=False)

my_survival_df

,case_id,case_submitter_id,died,time
0,0004d251-3f70-4395-b175-c94c2f5b1b81,TCGA-DD-AAVP,False,2752.0
2,000d566c-96c7-4f1c-b36e-fa2222467983,TCGA-KK-A7B2,False,1099.0
4,001887aa-36d0-463f-8bca-dec7043b4f2e,TCGA-DD-A4NP,False,3308.0
6,0024ab57-4036-4b0f-b7a1-040f97787022,TCGA-S7-A7WN,False,1133.0
8,002724fa-7051-49fa-9c58-4bcb7eba4ac6,TCGA-BG-A0M6,True,671
...,...,...,...,...
7699,ffafcbfd-0eb3-4e14-acd0-4407b641a4a5,TCGA-4Z-AA7Q,True,510
7701,ffbef57b-3627-4d6d-8857-a90e83911480,TCGA-TT-A6YJ,False,132.0
7703,ffcfa005-a04f-458e-9d1d-86143dd823e5,TCGA-LN-A4A6,False,391.0
7705,ffd8d31f-bc4b-4e19-bbaf-0e26e9f3a107,TCGA-XS-A8TJ,False,890.0


In [16]:
# Gene expression
gene_names = np.loadtxt('data/cancer_genes.txt', dtype=str)

def read_rnaseq(sample_id, count_type = 'fpkm_uq_unstranded'):
    # count_type = unstranded, stranded_first, stranded_second, tpm_unstranded, fpkm_unstranded, fpkm_uq_unstranded
    sample_file_df = tcga_file_df[tcga_file_df['Sample ID'] == sample_id]
    if len(sample_file_df) < 1:
        return
    if len(sample_file_df) > 1:
        display(sample_file_df)
        raise ValueError(f"{sample_id} has multiple files")
    file_id = sample_file_df['File ID'].item()
    file_name = sample_file_df['File Name'].item()
    filepath = os.path.join(in_dir, 'rnaseq', file_id, file_name)
    df = pd.read_csv(filepath, sep='\t', header=1)
    df = df.drop_duplicates(subset='gene_name', keep='last')
    df = df[df['gene_name'].isin(gene_names)]
    df = df[['gene_name', count_type]]
    df[count_type] = 1e4 / df[count_type].sum() * df[count_type]
    df[count_type] = np.log10(df[count_type] + 1e-3)
    df.index = df['gene_name'].values
    df = df.drop(columns=['gene_name'])
    df = df.T.reset_index()
    df = df.drop(columns=['index'])
    df['sample_id'] = sample_id
    df = df[['sample_id'] + df.columns.tolist()[:-1]]
    return df

expression_rows = []
for i, sample_id in my_sample_df['sample_submitter_id'].items():
    print(len(my_sample_df), i, end='\r')
    expression_rows.append(read_rnaseq(sample_id))
my_expression_df = pd.concat(expression_rows)
my_expression_df.to_csv(os.path.join(out_dir, 'transcriptomic_features.csv'), index=False)

my_expression_df.head()

,sample_id,LASP1,HOXA11,CREBBP,ETV1,CCL26,CD79B,PAX7,BTK,BRCA1,...,NUTM2D,AC129492.1,DDX3X,FANCG,SSX2,ETV5,CEBPA,LSM14A,CUX1,AC124319.1
0,TCGA-NC-A5HJ-01A,1.285341,-3.000000,0.727469,0.607738,-0.982086,0.192504,-2.177889,0.676704,0.343060,...,-1.129394,-1.089206,1.215301,-0.375599,-3.0,1.234440,0.706219,1.641721,0.811684,-0.854180
0,TCGA-XF-AAN2-01A,1.536252,-1.126228,0.460703,-0.872416,-0.974576,-0.642386,-1.789340,-0.247271,0.512107,...,-1.818193,-0.958073,1.287612,-0.599829,-3.0,0.149909,0.191165,1.439539,0.803881,0.382628
0,TCGA-CC-A8HS-01A,1.735546,-1.029349,0.912036,-0.336917,-0.065700,0.161745,-3.000000,-0.357813,0.467509,...,-0.011243,-3.000000,0.972206,-0.053285,-3.0,0.612519,2.141567,1.758316,0.729652,-0.896123
0,TCGA-EJ-A7NJ-01A,1.080870,0.458711,0.769942,-0.424266,-1.676778,-0.371588,-2.499555,-0.488597,-0.504200,...,-0.491598,-1.023119,1.018113,-1.162665,-3.0,0.475485,0.535011,1.346894,0.889783,-0.488130
0,TCGA-A2-A04Q-01A,1.205623,-0.391889,0.534550,-0.058256,-0.323373,0.692528,-2.554469,0.555039,0.079089,...,-1.715497,-1.697057,1.151251,-0.518796,-3.0,0.510935,0.427774,1.467141,0.514222,-0.576925
